# nanowhale 🐳 — 1B MoE from Scratch on Colab

**All-in-one notebook.** No external scripts needed.

- DeepSeek-V4 architecture (MLA + MoE + Hyper-Connections)
- ~1.2B total / ~400M active params (12 experts, top-2 routing)
- 256k context via curriculum training
- Reasoning-heavy dataset (code + math + logic)
- **Runs on Colab H100 / A100 80GB**

> ⚠️ **WIP — Do not use yet. Training stability still being refined.**

## 1. Setup

In [ ]:
!pip install -q torch transformers datasets safetensors pyyaml

import os, sys, math, random, json
from typing import Optional, Tuple, List
from functools import lru_cache

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import PreTrainedTokenizerFast
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast
from transformers.modeling_utils import PreTrainedModel
from transformers.generation import GenerationMixin
from transformers.configuration_utils import PretrainedConfig
from safetensors.torch import save_file, load_file

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")

## 2. Model Code — DeepSeek-V4 Architecture

In [ ]:
# ====================================================================
# Configuration
# ====================================================================

class DeepseekV4Config(PretrainedConfig):
    model_type = "deepseek_v4"
    keys_to_ignore_at_inference = ["past_key_values"]

    def __init__(
        self,
        vocab_size=128000,
        hidden_size=768,
        num_hidden_layers=16,
        num_attention_heads=16,
        num_key_value_heads=1,
        moe_intermediate_size=2048,
        n_routed_experts=12,
        n_shared_experts=2,
        num_experts_per_tok=2,
        norm_topk_prob=True,
        scoring_func="sqrtsoftplus",
        routed_scaling_factor=1.0,
        topk_method="noaux_tc",
        num_hash_layers=2,
        swiglu_limit=10.0,
        q_lora_rank=384,
        head_dim=128,
        qk_rope_head_dim=32,
        o_groups=4,
        o_lora_rank=192,
        sliding_window=128,
        compress_ratios=None,
        compress_rope_theta=160000.0,
        index_n_heads=64,
        index_head_dim=128,
        index_topk=512,
        hc_mult=2,
        hc_sinkhorn_iters=3,
        hc_eps=1e-6,
        num_nextn_predict_layers=1,
        hidden_act="silu",
        max_position_embeddings=4096,
        initializer_range=0.02,
        rms_norm_eps=1e-6,
        use_cache=True,
        pad_token_id=0,
        bos_token_id=0,
        eos_token_id=1,
        tie_word_embeddings=False,
        rope_theta=10000.0,
        rope_scaling=None,
        attention_bias=False,
        attention_dropout=0.0,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads or num_attention_heads
        self.moe_intermediate_size = moe_intermediate_size
        self.n_routed_experts = n_routed_experts
        self.n_shared_experts = n_shared_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.norm_topk_prob = norm_topk_prob
        self.scoring_func = scoring_func
        self.routed_scaling_factor = routed_scaling_factor
        self.topk_method = topk_method
        self.num_hash_layers = num_hash_layers
        self.swiglu_limit = swiglu_limit
        self.q_lora_rank = q_lora_rank
        self.head_dim = head_dim
        self.qk_rope_head_dim = qk_rope_head_dim
        self.nope_head_dim = head_dim - qk_rope_head_dim
        self.o_groups = o_groups
        self.o_lora_rank = o_lora_rank
        self.sliding_window = sliding_window
        if compress_ratios is None:
            compress_ratios = [0] * (num_hidden_layers + 1)
        self.compress_ratios = compress_ratios
        self.compress_rope_theta = compress_rope_theta
        self.index_n_heads = index_n_heads
        self.index_head_dim = index_head_dim
        self.index_topk = index_topk
        self.hc_mult = hc_mult
        self.hc_sinkhorn_iters = hc_sinkhorn_iters
        self.hc_eps = hc_eps
        self.num_nextn_predict_layers = num_nextn_predict_layers
        self.hidden_act = hidden_act
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = use_cache
        self.rope_theta = rope_theta
        self.rope_scaling = rope_scaling
        self.attention_bias = attention_bias
        self.attention_dropout = attention_dropout
        super().__init__(
            pad_token_id=pad_token_id, bos_token_id=bos_token_id,
            eos_token_id=eos_token_id, tie_word_embeddings=tie_word_embeddings, **kwargs,
        )

In [ ]:
# ====================================================================
# Utilities
# ====================================================================

class DeepseekV4RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        var = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(var + self.eps)
        return (self.weight * x).to(dtype)


def precompute_freqs_cis(dim, seqlen, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
    t = torch.arange(seqlen, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    cos = freqs.cos()
    sin = freqs.sin()
    return torch.stack([cos, sin], dim=0)


def apply_rotary_emb(x, cos_sin):
    cos, sin = cos_sin[0], cos_sin[1]
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    while cos.ndim < x1.ndim:
        cos, sin = cos.unsqueeze(0), sin.unsqueeze(0)
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return torch.cat([y1, y2], dim=-1).to(x.dtype)


def hc_split_sinkhorn(mixes, hc_scale, hc_base, hc_mult=2, sinkhorn_iters=3, eps=1e-6):
    pre_raw = mixes[..., :hc_mult]
    post_raw = mixes[..., hc_mult:2*hc_mult]
    comb_raw = mixes[..., 2*hc_mult:].reshape(*mixes.shape[:-1], hc_mult, hc_mult)
    pre = torch.sigmoid(pre_raw * hc_scale[0] + hc_base[:hc_mult]) + eps
    post = 2 * torch.sigmoid(post_raw * hc_scale[1] + hc_base[hc_mult:2*hc_mult])
    comb = comb_raw * hc_scale[2] + hc_base[2*hc_mult:].reshape(hc_mult, hc_mult)
    comb = F.softmax(comb, dim=-1) + eps
    comb = comb / (comb.sum(dim=-2, keepdim=True) + eps)
    for _ in range(sinkhorn_iters - 1):
        comb = comb / (comb.sum(dim=-1, keepdim=True) + eps)
        comb = comb / (comb.sum(dim=-2, keepdim=True) + eps)
    return pre, post, comb

In [ ]:
# ====================================================================
# Attention (MLA)
# ====================================================================

class DeepseekV4Attention(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.head_dim
        self.qk_rope_head_dim = config.qk_rope_head_dim
        self.nope_head_dim = config.head_dim - config.qk_rope_head_dim
        self.q_lora_rank = config.q_lora_rank
        self.o_groups = config.o_groups
        self.o_lora_rank = config.o_lora_rank
        self.scaling = config.head_dim ** -0.5

        self.wq_a = nn.Linear(self.hidden_size, self.q_lora_rank, bias=False)
        self.q_norm = DeepseekV4RMSNorm(self.q_lora_rank, config.rms_norm_eps)
        self.wq_b = nn.Linear(self.q_lora_rank, self.num_heads * self.head_dim, bias=False)
        self.wkv = nn.Linear(self.hidden_size, self.head_dim, bias=False)
        self.kv_norm = DeepseekV4RMSNorm(self.head_dim, config.rms_norm_eps)
        group_head_dim = self.num_heads * self.head_dim // self.o_groups
        self.wo_a = nn.Linear(group_head_dim, self.o_groups * self.o_lora_rank, bias=False)
        self.wo_b = nn.Linear(self.o_groups * self.o_lora_rank, self.hidden_size, bias=False)

    def forward(self, hidden_states, attention_mask=None, position_ids=None, freqs_cis=None,
                past_key_value=None, use_cache=False):
        bsz, seqlen, _ = hidden_states.shape
        q = self.q_norm(self.wq_a(hidden_states))
        q = self.wq_b(q)
        q = q.view(bsz, seqlen, self.num_heads, self.head_dim).transpose(1, 2)
        q = q * torch.rsqrt(q.float().pow(2).mean(-1, keepdim=True) + self.config.rms_norm_eps)
        q = q.to(hidden_states.dtype)
        kv = self.kv_norm(self.wkv(hidden_states)).unsqueeze(1)

        if freqs_cis is not None:
            q_rope = q[..., -self.qk_rope_head_dim:]
            kv_rope = kv[..., -self.qk_rope_head_dim:]
            q_rope = apply_rotary_emb(q_rope, freqs_cis)
            kv_rope = apply_rotary_emb(kv_rope, freqs_cis)
            q = torch.cat([q[..., :-self.qk_rope_head_dim], q_rope], dim=-1)
            kv = torch.cat([kv[..., :-self.qk_rope_head_dim], kv_rope], dim=-1)

        if past_key_value is not None:
            past_k, _ = past_key_value
            kv = torch.cat([past_k, kv], dim=2)
        new_cache = (kv, kv) if use_cache else None

        kv_expanded = kv.expand(-1, self.num_heads, -1, -1)
        attn_output = F.scaled_dot_product_attention(
            q, kv_expanded, kv_expanded, attn_mask=attention_mask,
            is_causal=(attention_mask is None), scale=self.scaling)

        if freqs_cis is not None:
            cos, sin = freqs_cis[0].unsqueeze(0).unsqueeze(0), -freqs_cis[1].unsqueeze(0).unsqueeze(0)
            out_rope = attn_output[..., -self.qk_rope_head_dim:]
            d = out_rope.shape[-1] // 2
            o1, o2 = out_rope[..., :d], out_rope[..., d:]
            out_rope = torch.cat([o1 * cos + o2 * sin, o1 * (-sin) + o2 * cos], dim=-1)
            attn_output = torch.cat([attn_output[..., :-self.qk_rope_head_dim], out_rope.to(attn_output.dtype)], dim=-1)

        attn_output = attn_output.transpose(1, 2).reshape(bsz, seqlen, self.o_groups, -1)
        wo_a_w = self.wo_a.weight.view(self.o_groups, self.o_lora_rank, -1)
        attn_output = torch.einsum("bsgd,grd->bsgr", attn_output, wo_a_w)
        attn_output = attn_output.flatten(2)
        attn_output = self.wo_b(attn_output)
        return attn_output, new_cache

In [ ]:
# ====================================================================
# MoE
# ====================================================================

class DeepseekV4Expert(nn.Module):
    def __init__(self, hidden_size, intermediate_size, swiglu_limit=10.0):
        super().__init__()
        self.w1 = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.w2 = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.w3 = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.swiglu_limit = swiglu_limit

    def forward(self, x):
        gate = self.w1(x).float()
        up = self.w3(x).float()
        if self.swiglu_limit > 0:
            up = up.clamp(-self.swiglu_limit, self.swiglu_limit)
            gate = gate.clamp(max=self.swiglu_limit)
        x = F.silu(gate) * up
        return self.w2(x.to(self.w2.weight.dtype))


class DeepseekV4Gate(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.topk = config.num_experts_per_tok
        self.scoring_func = config.scoring_func
        self.route_scale = config.routed_scaling_factor
        self.is_hash_layer = layer_idx < config.num_hash_layers
        self.weight = nn.Parameter(torch.empty(config.n_routed_experts, config.hidden_size))
        if not self.is_hash_layer:
            self.bias = nn.Parameter(torch.zeros(config.n_routed_experts))
        else:
            self.register_parameter("bias", None)

    def forward(self, x):
        scores = F.linear(x.float(), self.weight.float())
        if self.scoring_func == "softmax":
            scores = scores.softmax(dim=-1)
        elif self.scoring_func == "sigmoid":
            scores = scores.sigmoid()
        elif self.scoring_func == "sqrtsoftplus":
            scores = F.softplus(scores).sqrt()
        original_scores = scores
        if self.bias is not None:
            scores = scores + self.bias
        indices = scores.topk(self.topk, dim=-1)[1]
        weights = original_scores.gather(1, indices)
        if self.scoring_func != "softmax":
            weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-20)
        weights = weights * self.route_scale
        return weights.to(x.dtype), indices


class DeepseekV4MoE(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.n_routed_experts = config.n_routed_experts
        self.gate = DeepseekV4Gate(config, layer_idx)
        self.experts = nn.ModuleList([
            DeepseekV4Expert(config.hidden_size, config.moe_intermediate_size, config.swiglu_limit)
            for _ in range(config.n_routed_experts)])
        self.shared_expert = DeepseekV4Expert(config.hidden_size, config.moe_intermediate_size)

    def forward(self, x):
        shape = x.shape
        x_flat = x.view(-1, self.hidden_size)
        weights, indices = self.gate(x_flat)
        y = torch.zeros_like(x_flat, dtype=torch.float32)
        counts = torch.bincount(indices.flatten(), minlength=self.n_routed_experts)
        for i in range(self.n_routed_experts):
            if counts[i] == 0:
                continue
            idx, top = torch.where(indices == i)
            expert_out = self.experts[i](x_flat[idx])
            y[idx] += (weights[idx, top].unsqueeze(-1) * expert_out.float())
        y = y + self.shared_expert(x_flat).float()
        return y.to(x.dtype).view(shape)


# ====================================================================
# Transformer Block with Hyper-Connections
# ====================================================================

class DeepseekV4Block(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.config = config
        self.hc_mult = config.hc_mult
        self.norm_eps = config.rms_norm_eps
        self.hc_eps = config.hc_eps
        self.hc_sinkhorn_iters = config.hc_sinkhorn_iters
        self.attn = DeepseekV4Attention(config, layer_idx)
        self.ffn = DeepseekV4MoE(config, layer_idx)
        self.attn_norm = DeepseekV4RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.ffn_norm = DeepseekV4RMSNorm(config.hidden_size, config.rms_norm_eps)
        mix_hc = (2 + config.hc_mult) * config.hc_mult
        hc_dim = config.hc_mult * config.hidden_size
        self.hc_attn_fn = nn.Parameter(torch.empty(mix_hc, hc_dim))
        self.hc_ffn_fn = nn.Parameter(torch.empty(mix_hc, hc_dim))
        self.hc_attn_base = nn.Parameter(torch.empty(mix_hc))
        self.hc_ffn_base = nn.Parameter(torch.empty(mix_hc))
        self.hc_attn_scale = nn.Parameter(torch.empty(3))
        self.hc_ffn_scale = nn.Parameter(torch.empty(3))

    def hc_pre(self, x, hc_fn, hc_scale, hc_base):
        dtype = x.dtype
        x_flat = x.flatten(2).float()
        rsqrt = torch.rsqrt(x_flat.pow(2).mean(-1, keepdim=True) + self.norm_eps)
        mixes = F.linear(x_flat, hc_fn.float()) * rsqrt
        pre, post, comb = hc_split_sinkhorn(
            mixes, hc_scale, hc_base, self.hc_mult, self.hc_sinkhorn_iters, self.hc_eps)
        y = (pre.unsqueeze(-1) * x.float()).sum(dim=2)
        return y.to(dtype), post, comb

    def hc_post(self, x, residual, post, comb):
        y = (post.unsqueeze(-1) * x.unsqueeze(2).float() +
             torch.einsum("bsij,bsjd->bsid", comb.float(), residual.float()))
        return y.to(x.dtype)

    def forward(self, x, attention_mask=None, position_ids=None, freqs_cis=None,
                past_key_value=None, use_cache=False):
        residual = x
        y, post, comb = self.hc_pre(x, self.hc_attn_fn, self.hc_attn_scale, self.hc_attn_base)
        y = self.attn_norm(y)
        y, new_cache = self.attn(y, attention_mask=attention_mask, position_ids=position_ids,
                                  freqs_cis=freqs_cis, past_key_value=past_key_value, use_cache=use_cache)
        x = self.hc_post(y, residual, post, comb)
        residual = x
        y, post, comb = self.hc_pre(x, self.hc_ffn_fn, self.hc_ffn_scale, self.hc_ffn_base)
        y = self.ffn_norm(y)
        y = self.ffn(y)
        x = self.hc_post(y, residual, post, comb)
        return x, new_cache

In [ ]:
# ====================================================================
# Full Model + CausalLM Head
# ====================================================================

class DeepseekV4PreTrainedModel(PreTrainedModel):
    config_class = DeepseekV4Config
    base_model_prefix = "model"
    supports_gradient_checkpointing = True
    _no_split_modules = ["DeepseekV4Block"]
    _skip_keys_device_placement = ["past_key_values"]

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=std)
        elif isinstance(module, DeepseekV4RMSNorm):
            module.weight.data.fill_(1.0)
        elif isinstance(module, DeepseekV4Gate):
            module.weight.data.normal_(mean=0.0, std=std)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, DeepseekV4Block):
            nn.init.normal_(module.hc_attn_fn, std=0.01)
            nn.init.normal_(module.hc_ffn_fn, std=0.01)
            nn.init.zeros_(module.hc_attn_base)
            nn.init.zeros_(module.hc_ffn_base)
            nn.init.ones_(module.hc_attn_scale)
            nn.init.ones_(module.hc_ffn_scale)
        elif isinstance(module, DeepseekV4Attention):
            pass


class DeepseekV4Model(DeepseekV4PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            DeepseekV4Block(config, i) for i in range(config.num_hidden_layers)])
        self.norm = DeepseekV4RMSNorm(config.hidden_size, config.rms_norm_eps)
        hc_dim = config.hc_mult * config.hidden_size
        self.hc_head_fn = nn.Parameter(torch.empty(config.hc_mult, hc_dim))
        self.hc_head_base = nn.Parameter(torch.empty(config.hc_mult))
        self.hc_head_scale = nn.Parameter(torch.empty(1))
        self.register_buffer("freqs_cis",
            precompute_freqs_cis(config.qk_rope_head_dim, config.max_position_embeddings, config.rope_theta),
            persistent=False)
        self.gradient_checkpointing = False
        self.post_init()

    def _init_weights(self, module):
        super()._init_weights(module)
        if module is self:
            nn.init.normal_(self.hc_head_fn, std=0.01)
            nn.init.zeros_(self.hc_head_base)
            nn.init.ones_(self.hc_head_scale)

    def hc_head(self, x):
        dtype = x.dtype
        x_flat = x.flatten(2).float()
        rsqrt = torch.rsqrt(x_flat.pow(2).mean(-1, keepdim=True) + self.config.rms_norm_eps)
        mixes = F.linear(x_flat, self.hc_head_fn.float()) * rsqrt
        pre = torch.sigmoid(mixes * self.hc_head_scale.float() + self.hc_head_base.float()) + self.config.hc_eps
        y = (pre.unsqueeze(-1) * x.float()).sum(dim=2)
        return y.to(dtype)

    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, use_cache=None,
                output_hidden_states=None, return_dict=None):
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        bsz, seqlen = inputs_embeds.shape[:2]
        if position_ids is None:
            position_ids = torch.arange(seqlen, device=inputs_embeds.device).unsqueeze(0)
        position_ids = position_ids.clamp(0, self.config.max_position_embeddings - 1)
        pos = position_ids.squeeze(0)
        freqs_cis = self.freqs_cis[:, pos].to(inputs_embeds.device)

        if attention_mask is None:
            causal_mask = torch.full((seqlen, seqlen), float("-inf"), device=inputs_embeds.device, dtype=inputs_embeds.dtype)
            causal_mask = torch.triu(causal_mask, diagonal=1).unsqueeze(0).unsqueeze(0)
        else:
            causal = torch.triu(torch.full((seqlen, seqlen), float("-inf"), device=inputs_embeds.device, dtype=inputs_embeds.dtype), diagonal=1)
            padding_mask = (1.0 - attention_mask) * float("-inf")
            padding_mask = padding_mask.unsqueeze(1).unsqueeze(2).to(dtype=inputs_embeds.dtype)
            causal_mask = causal.unsqueeze(0).unsqueeze(0) + padding_mask

        hidden_states = inputs_embeds.unsqueeze(2).expand(-1, -1, self.config.hc_mult, -1).contiguous()
        for layer in self.layers:
            if self.gradient_checkpointing and self.training:
                hidden_states, _ = torch.utils.checkpoint.checkpoint(
                    layer, hidden_states, causal_mask, position_ids, freqs_cis, None, False, use_reentrant=False)
            else:
                hidden_states, _ = layer(hidden_states, attention_mask=causal_mask, position_ids=position_ids,
                                          freqs_cis=freqs_cis, past_key_value=None, use_cache=False)
        hidden_states = self.hc_head(hidden_states)
        hidden_states = self.norm(hidden_states)
        return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=None)


class DeepseekV4ForCausalLM(DeepseekV4PreTrainedModel, GenerationMixin):
    _tied_weights_keys = {"lm_head.weight": "model.embed_tokens.weight"}

    def __init__(self, config):
        super().__init__(config)
        self.model = DeepseekV4Model(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, labels=None, use_cache=None,
                output_hidden_states=None, return_dict=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask,
                            position_ids=position_ids, past_key_values=past_key_values,
                            inputs_embeds=inputs_embeds, use_cache=use_cache,
                            output_hidden_states=output_hidden_states, return_dict=True)
        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, self.config.vocab_size),
                                   shift_labels.view(-1), ignore_index=-100)
        return CausalLMOutputWithPast(loss=loss, logits=logits, past_key_values=None)

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, **kwargs):
        if past_key_values is not None:
            input_ids = input_ids[:, -1:]
        return {"input_ids": input_ids, "past_key_values": past_key_values, "use_cache": True}

print("Model architecture loaded.")

## 3. Download DeepSeek-V4 Tokenizer

In [ ]:
# Download the DeepSeek-V4 tokenizer from HuggingFace
!mkdir -p tokenizer
!wget -q -O tokenizer/tokenizer.json https://huggingface.co/cmpatino/nanowhale-100m/resolve/main/tokenizer.json
!wget -q -O tokenizer/tokenizer_config.json https://huggingface.co/cmpatino/nanowhale-100m/resolve/main/tokenizer_config.json

tokenizer = PreTrainedTokenizerFast.from_pretrained("tokenizer")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

## 4. Prepare Reasoning Dataset

High-quality public datasets targeting english, code, math, and reasoning.

In [ ]:
DATASETS = [
    # (key, name, weight, min_chars, purpose)
    ("fineweb2", "HuggingFaceFW/fineweb-2", 0.28, 300, "general-knowledge"),
    ("c4",       "allenai/c4",              0.20, 350, "clean-web"),
    ("fffweb",   "m-a-p/FineFineWeb",       0.12, 400, "curated-web"),
    ("code",     "deepmind/code_contests",  0.12, 128, "coding"),
    ("mbpp",     "google-research-datasets/mbpp", 0.08, 64, "python"),
    ("gsm8k",    "openai/gsm8k",            0.12, 64,  "math"),
    ("long",     "HuggingFaceFW/fineweb-2", 0.08, 500, "long-context"),
]

MAX_PER = 8000
PACK_LONG = 49152
all_examples = []

for key, name, weight, min_chars, purpose in DATASETS:
    print(f"[{key}] {purpose} (weight={weight})...")
    try:
        ds = load_dataset(name, split="train", streaming=True)
    except:
        try:
            ds = load_dataset(name, "en", split="train", streaming=True)
        except:
            try:
                ds = load_dataset(name, "main", split="train", streaming=False)
            except:
                print(f"  SKIP: cannot load")
                continue

    samples = []
    for s in ds:
        text = s.get("text") or s.get("content") or ""
        if "question" in s:
            a = s.get("answer", "") or s.get("solution", "")
            text = f"Q: {s['question']}\nA: {a}" if a else s["question"]
        if "problem" in s:
            sol = s.get("solution", "") or s.get("solutions", "")
            text = f"Problem: {s['problem']}\nSolution: {sol}" if sol else s["problem"]
        if not text or len(text) < min_chars:
            continue
        samples.append({"text": text})
        if len(samples) >= MAX_PER:
            break

    if key == "long":
        packed, buf = [], ""
        for s in samples:
            buf += "\n\n" + s["text"]
            if len(buf) > PACK_LONG:
                packed.append({"text": buf.strip()}); buf = ""
        if buf: packed.append({"text": buf.strip()})
        samples = packed
        print(f"  Packed {len(samples)} long-context examples")
    else:
        print(f"  {len(samples)} examples")

    all_examples.extend(samples)

random.seed(42)
random.shuffle(all_examples)
print(f"\nTotal: {len(all_examples):,} examples")

## 5. Build Model (~1.2B params)

In [ ]:
config = DeepseekV4Config(
    vocab_size=128000,
    hidden_size=768,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=1,
    head_dim=128,
    qk_rope_head_dim=32,
    q_lora_rank=384,
    o_groups=4,
    o_lora_rank=192,
    moe_intermediate_size=2048,
    n_routed_experts=12,
    n_shared_experts=2,
    num_experts_per_tok=2,
    hc_mult=2,
    hc_sinkhorn_iters=3,
    max_position_embeddings=262144,
    rope_theta=500000.0,
)

model = DeepseekV4ForCausalLM(config).cuda()
total = sum(p.numel() for p in model.parameters())
print(f"Model: {total:,} params ({total/1e9:.2f}B)")

# torch.compile for H100 speedup
try:
    model = torch.compile(model, mode="reduce-overhead")
    print("torch.compile: ON")
except Exception as e:
    print(f"torch.compile: SKIP ({e})")

## 6. Training Loop

Curriculum context: 4k → 8k → 32k → 64k → 128k → 256k

In [ ]:
# === Training settings ===
STEPS = 25000            # Full run (start with 200 for smoke test)
LR = 3e-4
GRAD_ACCUM = 16
LOG_EVERY = 20
SAVE_EVERY = 5000
OUTPUT_DIR = "/content/checkpoints/nanowhale_1b"

# Curriculum: (step, seq_len)
CURRICULUM = [(0, 4096), (1000, 8192), (3000, 32768), (8000, 65536), (15000, 131072), (25000, 262144)]
# Override with short for smoke test
if STEPS <= 500:
    CURRICULUM = [(0, 2048)]

os.makedirs(OUTPUT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1,
                              betas=(0.9, 0.95), fused=True)
scaler = torch.amp.GradScaler("cuda")

def get_cur_len(step):
    cur = CURRICULUM[0][1]
    for s, l in CURRICULUM:
        if step >= s: cur = l
    return cur

print(f"Training {STEPS} steps on {len(all_examples)} examples")
print(f"Effective batch: {GRAD_ACCUM}")
print(f"Curriculum: {CURRICULUM}")

In [ ]:
model.train()
losses = []
cur_len = get_cur_len(0)
acc_loss = 0.0
micro = 0

# Pre-tokenize at initial context length
def tokenize_batch(examples, max_len):
    out = []
    for ex in examples:
        ids = tokenizer.encode(ex["text"], truncation=True, max_length=max_len)
        out.append({"input_ids": ids, "labels": ids})
    return out

tokenized = tokenize_batch(all_examples, cur_len)

for step in range(STEPS):
    new_len = get_cur_len(step)
    if new_len != cur_len:
        cur_len = new_len
        tokenized = tokenize_batch(all_examples, cur_len)
        print(f"  [Curriculum] Step {step}: seq_len -> {cur_len}")

    ex = tokenized[step % len(tokenized)]
    if len(ex["input_ids"]) < 16:
        continue

    alen = min(len(ex["input_ids"]) - 1, cur_len)
    inp = torch.tensor([ex["input_ids"][:alen]], dtype=torch.long).cuda()
    lbl = torch.tensor([ex["labels"][1:alen+1]], dtype=torch.long).cuda()

    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=inp, labels=lbl)
        loss = out.loss / GRAD_ACCUM

    scaler.scale(loss).backward()
    acc_loss += loss.item()
    micro += 1

    if micro % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        losses.append(acc_loss * GRAD_ACCUM)
        acc_loss = 0.0

    if (step + 1) % LOG_EVERY == 0 and losses:
        avg = sum(losses[-LOG_EVERY:]) / min(len(losses), LOG_EVERY)
        print(f"  Step {step+1:5d} | Loss: {avg:.4f} | Seq: {cur_len}")

    if (step + 1) % SAVE_EVERY == 0:
        ckpt = os.path.join(OUTPUT_DIR, f"step_{step+1}")
        os.makedirs(ckpt, exist_ok=True)
        save_file(model.state_dict(), os.path.join(ckpt, "model.safetensors"))
        torch.save({"optimizer": optimizer.state_dict(), "step": step+1},
                   os.path.join(ckpt, "optimizer.pt"))
        tokenizer.save_pretrained(ckpt)
        print(f"  [Saved] {ckpt}")

In [ ]:
# === Final save ===
final = os.path.join(OUTPUT_DIR, "final")
os.makedirs(final, exist_ok=True)
save_file(model.state_dict(), os.path.join(final, "model.safetensors"))
tokenizer.save_pretrained(final)

print("="*60)
print(f"DONE. Model saved to {final}")
if losses:
    print(f"Final loss: {sum(losses[-100:])/len(losses[-100:]):.4f}")
print("="*60)

## 7. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
src = "/content/checkpoints/nanowhale_1b/final"
dst = "/content/drive/MyDrive/nanowhale-1b-moe"
if os.path.exists(src):
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Saved to Google Drive: {dst}")
else:
    print(f"No checkpoint at {src} — training may not have completed")

## 8. Quick Inference Test

In [ ]:
ckpt_path = os.path.join(OUTPUT_DIR, "final", "model.safetensors")
if os.path.exists(ckpt_path):
    state = load_file(ckpt_path)
    model.load_state_dict(state, strict=False)
    model.eval()

    prompt = "def solve_quadratic(a, b, c):"
    inp = tokenizer.encode(prompt, return_tensors="pt").cuda()
    with torch.no_grad():
        out = model.generate(inp, max_new_tokens=150, temperature=0.7, do_sample=True)
    print(tokenizer.decode(out[0]))
else:
    print("No checkpoint found — run training first")